# 06 — Train Sgan / 生成对抗网络

**Chapter 20 — File 6 of 7 / 第20章 — 第6个文件（共7个）**

---

## Summary / 总结

This script demonstrates **example of semi-supervised gan for mnist**.

本脚本演示 **example of semi-supervised gan for mnist**。

---
## Step 1 — example of semi-supervised gan for mnist

In [ ]:
from numpy import expand_dims
from numpy import zeros
from numpy import ones
from numpy import asarray
from numpy.random import randn
from numpy.random import randint
from keras.datasets.mnist import load_data
from keras.optimizers import Adam
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Reshape
from keras.layers import Flatten
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.layers import Dropout
from keras.layers import Lambda
from keras.layers import Activation
from matplotlib import pyplot
from keras import backend

---
## Step 2 — custom activation function

In [ ]:
def custom_activation(output):
	logexpsum = backend.sum(backend.exp(output), axis=-1, keepdims=True)
	result = logexpsum / (logexpsum + 1.0)
	return result

---
## Step 3 — define the standalone supervised and unsupervised discriminator models

In [ ]:
def define_discriminator(in_shape=(28,28,1), n_classes=10):

---
## Step 4 — image input

In [ ]:
in_image = Input(shape=in_shape)

---
## Step 5 — downsample

In [ ]:
fe = Conv2D(128, (3,3), strides=(2,2), padding='same')(in_image)
	fe = LeakyReLU(alpha=0.2)(fe)

---
## Step 6 — downsample

In [ ]:
fe = Conv2D(128, (3,3), strides=(2,2), padding='same')(fe)
	fe = LeakyReLU(alpha=0.2)(fe)

---
## Step 7 — downsample

In [ ]:
fe = Conv2D(128, (3,3), strides=(2,2), padding='same')(fe)
	fe = LeakyReLU(alpha=0.2)(fe)

---
## Step 8 — flatten feature maps

In [ ]:
fe = Flatten()(fe)

---
## Step 9 — dropout

In [ ]:
fe = Dropout(0.4)(fe)

---
## Step 10 — output layer nodes

In [ ]:
fe = Dense(n_classes)(fe)

---
## Step 11 — supervised output

In [ ]:
c_out_layer = Activation('softmax')(fe)

---
## Step 12 — define and compile supervised discriminator model

In [ ]:
c_model = Model(in_image, c_out_layer)
	c_model.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(lr=0.0002, beta_1=0.5), metrics=['accuracy'])

---
## Step 13 — unsupervised output

In [ ]:
d_out_layer = Lambda(custom_activation)(fe)

---
## Step 14 — define and compile unsupervised discriminator model

In [ ]:
d_model = Model(in_image, d_out_layer)
	d_model.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.0002, beta_1=0.5))
	return d_model, c_model

---
## Step 15 — define the standalone generator model

In [ ]:
def define_generator(latent_dim):

---
## Step 16 — image generator input

In [ ]:
in_lat = Input(shape=(latent_dim,))

---
## Step 17 — foundation for 7x7 image

In [ ]:
n_nodes = 128 * 7 * 7
	gen = Dense(n_nodes)(in_lat)
	gen = LeakyReLU(alpha=0.2)(gen)
	gen = Reshape((7, 7, 128))(gen)

---
## Step 18 — upsample to 14x14

In [ ]:
gen = Conv2DTranspose(128, (4,4), strides=(2,2), padding='same')(gen)
	gen = LeakyReLU(alpha=0.2)(gen)

---
## Step 19 — upsample to 28x28

In [ ]:
gen = Conv2DTranspose(128, (4,4), strides=(2,2), padding='same')(gen)
	gen = LeakyReLU(alpha=0.2)(gen)

---
## Step 20 — output

In [ ]:
out_layer = Conv2D(1, (7,7), activation='tanh', padding='same')(gen)

---
## Step 21 — define model

In [ ]:
model = Model(in_lat, out_layer)
	return model

---
## Step 22 — define the combined generator and discriminator model, for updating the generator

In [ ]:
def define_gan(g_model, d_model):

---
## Step 23 — make weights in the discriminator not trainable

In [ ]:
d_model.trainable = False

---
## Step 24 — connect image output from generator as input to discriminator

In [ ]:
gan_output = d_model(g_model.output)

---
## Step 25 — define gan model as taking noise and outputting a classification

In [ ]:
model = Model(g_model.input, gan_output)

---
## Step 26 — compile model

In [ ]:
opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt)
	return model

---
## Step 27 — load the images

In [ ]:
def load_real_samples():

---
## Step 28 — load dataset

In [ ]:
(trainX, trainy), (_, _) = load_data()

---
## Step 29 — expand to 3d, e.g. add channels

In [ ]:
X = expand_dims(trainX, axis=-1)

---
## Step 30 — convert from ints to floats

In [ ]:
X = X.astype('float32')

---
## Step 31 — scale from [0,255] to [-1,1]

In [ ]:
X = (X - 127.5) / 127.5
	print(X.shape, trainy.shape)
	return [X, trainy]

---
## Step 32 — select a supervised subset of the dataset, ensures classes are balanced

In [ ]:
def select_supervised_samples(dataset, n_samples=100, n_classes=10):
	X, y = dataset
	X_list, y_list = list(), list()
	n_per_class = int(n_samples / n_classes)
	for i in range(n_classes):

---
## Step 33 — get all images for this class

In [ ]:
X_with_class = X[y == i]

---
## Step 34 — choose random instances

In [ ]:
ix = randint(0, len(X_with_class), n_per_class)

---
## Step 35 — add to list

In [ ]:
[X_list.append(X_with_class[j]) for j in ix]
		[y_list.append(i) for j in ix]
	return asarray(X_list), asarray(y_list)

---
## Step 36 — select real samples

In [ ]:
def generate_real_samples(dataset, n_samples):

---
## Step 37 — split into images and labels

In [ ]:
images, labels = dataset

---
## Step 38 — choose random instances

In [ ]:
ix = randint(0, images.shape[0], n_samples)

---
## Step 39 — select images and labels

In [ ]:
X, labels = images[ix], labels[ix]

---
## Step 40 — generate class labels

In [ ]:
y = ones((n_samples, 1))
	return [X, labels], y

---
## Step 41 — generate points in latent space as input for the generator

In [ ]:
def generate_latent_points(latent_dim, n_samples):

---
## Step 42 — generate points in the latent space

In [ ]:
z_input = randn(latent_dim * n_samples)

---
## Step 43 — reshape into a batch of inputs for the network

In [ ]:
z_input = z_input.reshape(n_samples, latent_dim)
	return z_input

---
## Step 44 — use the generator to generate n fake examples, with class labels

In [ ]:
def generate_fake_samples(generator, latent_dim, n_samples):

---
## Step 45 — generate points in latent space

In [ ]:
z_input = generate_latent_points(latent_dim, n_samples)

---
## Step 46 — predict outputs

In [ ]:
images = generator.predict(z_input)

---
## Step 47 — create class labels

In [ ]:
y = zeros((n_samples, 1))
	return images, y

---
## Step 48 — generate samples and save as a plot and save the model

In [ ]:
def summarize_performance(step, g_model, c_model, latent_dim, dataset, n_samples=100):

---
## Step 49 — prepare fake examples

In [ ]:
X, _ = generate_fake_samples(g_model, latent_dim, n_samples)

---
## Step 50 — scale from [-1,1] to [0,1]

In [ ]:
X = (X + 1) / 2.0

---
## Step 51 — plot images

In [ ]:
for i in range(100):

---
## Step 52 — define subplot

In [ ]:
pyplot.subplot(10, 10, 1 + i)

---
## Step 53 — turn off axis

In [ ]:
pyplot.axis('off')

---
## Step 54 — plot raw pixel data

In [ ]:
pyplot.imshow(X[i, :, :, 0], cmap='gray_r')

---
## Step 55 — save plot to file

In [ ]:
filename1 = 'generated_plot_%04d.png' % (step+1)
	pyplot.savefig(filename1)
	pyplot.close()

---
## Step 56 — evaluate the classifier model

In [ ]:
X, y = dataset
	_, acc = c_model.evaluate(X, y, verbose=0)
	print('Classifier Accuracy: %.3f%%' % (acc * 100))

---
## Step 57 — save the generator model

In [ ]:
filename2 = 'g_model_%04d.h5' % (step+1)
	g_model.save(filename2)

---
## Step 58 — save the classifier model

In [ ]:
filename3 = 'c_model_%04d.h5' % (step+1)
	c_model.save(filename3)
	print('>Saved: %s, %s, and %s' % (filename1, filename2, filename3))

---
## Step 59 — train the generator and discriminator

In [ ]:
def train(g_model, d_model, c_model, gan_model, dataset, latent_dim, n_epochs=20, n_batch=100):

---
## Step 60 — select supervised dataset

In [ ]:
X_sup, y_sup = select_supervised_samples(dataset)
	print(X_sup.shape, y_sup.shape)

---
## Step 61 — calculate the number of batches per training epoch

In [ ]:
bat_per_epo = int(dataset[0].shape[0] / n_batch)

---
## Step 62 — calculate the number of training iterations

In [ ]:
n_steps = bat_per_epo * n_epochs

---
## Step 63 — calculate the size of half a batch of samples

In [ ]:
half_batch = int(n_batch / 2)
	print('n_epochs=%d, n_batch=%d, 1/2=%d, b/e=%d, steps=%d' % (n_epochs, n_batch, half_batch, bat_per_epo, n_steps))

---
## Step 64 — manually enumerate epochs

In [ ]:
for i in range(n_steps):

---
## Step 65 — update supervised discriminator (c)

In [ ]:
[Xsup_real, ysup_real], _ = generate_real_samples([X_sup, y_sup], half_batch)
		c_loss, c_acc = c_model.train_on_batch(Xsup_real, ysup_real)

---
## Step 66 — update unsupervised discriminator (d)

In [ ]:
[X_real, _], y_real = generate_real_samples(dataset, half_batch)
		d_loss1 = d_model.train_on_batch(X_real, y_real)
		X_fake, y_fake = generate_fake_samples(g_model, latent_dim, half_batch)
		d_loss2 = d_model.train_on_batch(X_fake, y_fake)

---
## Step 67 — update generator (g)

In [ ]:
X_gan, y_gan = generate_latent_points(latent_dim, n_batch), ones((n_batch, 1))
		g_loss = gan_model.train_on_batch(X_gan, y_gan)

---
## Step 68 — summarize loss on this batch

In [ ]:
print('>%d, c[%.3f,%.0f], d[%.3f,%.3f], g[%.3f]' % (i+1, c_loss, c_acc*100, d_loss1, d_loss2, g_loss))

---
## Step 69 — evaluate the model performance every so often

In [ ]:
if (i+1) % (bat_per_epo * 1) == 0:
			summarize_performance(i, g_model, c_model, latent_dim, dataset)

---
## Step 70 — size of the latent space

In [ ]:
latent_dim = 100

---
## Step 71 — create the discriminator models

In [ ]:
d_model, c_model = define_discriminator()

---
## Step 72 — create the generator

In [ ]:
g_model = define_generator(latent_dim)

---
## Step 73 — create the gan

In [ ]:
gan_model = define_gan(g_model, d_model)

---
## Step 74 — load image data

In [ ]:
dataset = load_real_samples()

---
## Step 75 — train model

In [ ]:
train(g_model, d_model, c_model, gan_model, dataset, latent_dim)

---
## Learning Notes / 学习笔记

- **概念**: example of semi-supervised gan for mnist 是机器学习中的常用技术。  
  *example of semi-supervised gan for mnist is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Train Sgan / 生成对抗网络
# Complete Code / 完整代码
# ===============================

# example of semi-supervised gan for mnist
from numpy import expand_dims
from numpy import zeros
from numpy import ones
from numpy import asarray
from numpy.random import randn
from numpy.random import randint
from keras.datasets.mnist import load_data
from keras.optimizers import Adam
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Reshape
from keras.layers import Flatten
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.layers import Dropout
from keras.layers import Lambda
from keras.layers import Activation
from matplotlib import pyplot
from keras import backend

# custom activation function
def custom_activation(output):
	logexpsum = backend.sum(backend.exp(output), axis=-1, keepdims=True)
	result = logexpsum / (logexpsum + 1.0)
	return result

# define the standalone supervised and unsupervised discriminator models
def define_discriminator(in_shape=(28,28,1), n_classes=10):
	# image input
	in_image = Input(shape=in_shape)
	# downsample
	fe = Conv2D(128, (3,3), strides=(2,2), padding='same')(in_image)
	fe = LeakyReLU(alpha=0.2)(fe)
	# downsample
	fe = Conv2D(128, (3,3), strides=(2,2), padding='same')(fe)
	fe = LeakyReLU(alpha=0.2)(fe)
	# downsample
	fe = Conv2D(128, (3,3), strides=(2,2), padding='same')(fe)
	fe = LeakyReLU(alpha=0.2)(fe)
	# flatten feature maps
	fe = Flatten()(fe)
	# dropout
	fe = Dropout(0.4)(fe)
	# output layer nodes
	fe = Dense(n_classes)(fe)
	# supervised output
	c_out_layer = Activation('softmax')(fe)
	# define and compile supervised discriminator model
	c_model = Model(in_image, c_out_layer)
	c_model.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(lr=0.0002, beta_1=0.5), metrics=['accuracy'])
	# unsupervised output
	d_out_layer = Lambda(custom_activation)(fe)
	# define and compile unsupervised discriminator model
	d_model = Model(in_image, d_out_layer)
	d_model.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.0002, beta_1=0.5))
	return d_model, c_model

# define the standalone generator model
def define_generator(latent_dim):
	# image generator input
	in_lat = Input(shape=(latent_dim,))
	# foundation for 7x7 image
	n_nodes = 128 * 7 * 7
	gen = Dense(n_nodes)(in_lat)
	gen = LeakyReLU(alpha=0.2)(gen)
	gen = Reshape((7, 7, 128))(gen)
	# upsample to 14x14
	gen = Conv2DTranspose(128, (4,4), strides=(2,2), padding='same')(gen)
	gen = LeakyReLU(alpha=0.2)(gen)
	# upsample to 28x28
	gen = Conv2DTranspose(128, (4,4), strides=(2,2), padding='same')(gen)
	gen = LeakyReLU(alpha=0.2)(gen)
	# output
	out_layer = Conv2D(1, (7,7), activation='tanh', padding='same')(gen)
	# define model
	model = Model(in_lat, out_layer)
	return model

# define the combined generator and discriminator model, for updating the generator
def define_gan(g_model, d_model):
	# make weights in the discriminator not trainable
	d_model.trainable = False
	# connect image output from generator as input to discriminator
	gan_output = d_model(g_model.output)
	# define gan model as taking noise and outputting a classification
	model = Model(g_model.input, gan_output)
	# compile model
	opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt)
	return model

# load the images
def load_real_samples():
	# load dataset
	(trainX, trainy), (_, _) = load_data()
	# expand to 3d, e.g. add channels
	X = expand_dims(trainX, axis=-1)
	# convert from ints to floats
	X = X.astype('float32')
	# scale from [0,255] to [-1,1]
	X = (X - 127.5) / 127.5
	print(X.shape, trainy.shape)
	return [X, trainy]

# select a supervised subset of the dataset, ensures classes are balanced
def select_supervised_samples(dataset, n_samples=100, n_classes=10):
	X, y = dataset
	X_list, y_list = list(), list()
	n_per_class = int(n_samples / n_classes)
	for i in range(n_classes):
		# get all images for this class
		X_with_class = X[y == i]
		# choose random instances
		ix = randint(0, len(X_with_class), n_per_class)
		# add to list
		[X_list.append(X_with_class[j]) for j in ix]
		[y_list.append(i) for j in ix]
	return asarray(X_list), asarray(y_list)

# select real samples
def generate_real_samples(dataset, n_samples):
	# split into images and labels
	images, labels = dataset
	# choose random instances
	ix = randint(0, images.shape[0], n_samples)
	# select images and labels
	X, labels = images[ix], labels[ix]
	# generate class labels
	y = ones((n_samples, 1))
	return [X, labels], y

# generate points in latent space as input for the generator
def generate_latent_points(latent_dim, n_samples):
	# generate points in the latent space
	z_input = randn(latent_dim * n_samples)
	# reshape into a batch of inputs for the network
	z_input = z_input.reshape(n_samples, latent_dim)
	return z_input

# use the generator to generate n fake examples, with class labels
def generate_fake_samples(generator, latent_dim, n_samples):
	# generate points in latent space
	z_input = generate_latent_points(latent_dim, n_samples)
	# predict outputs
	images = generator.predict(z_input)
	# create class labels
	y = zeros((n_samples, 1))
	return images, y

# generate samples and save as a plot and save the model
def summarize_performance(step, g_model, c_model, latent_dim, dataset, n_samples=100):
	# prepare fake examples
	X, _ = generate_fake_samples(g_model, latent_dim, n_samples)
	# scale from [-1,1] to [0,1]
	X = (X + 1) / 2.0
	# plot images
	for i in range(100):
		# define subplot
		pyplot.subplot(10, 10, 1 + i)
		# turn off axis
		pyplot.axis('off')
		# plot raw pixel data
		pyplot.imshow(X[i, :, :, 0], cmap='gray_r')
	# save plot to file
	filename1 = 'generated_plot_%04d.png' % (step+1)
	pyplot.savefig(filename1)
	pyplot.close()
	# evaluate the classifier model
	X, y = dataset
	_, acc = c_model.evaluate(X, y, verbose=0)
	print('Classifier Accuracy: %.3f%%' % (acc * 100))
	# save the generator model
	filename2 = 'g_model_%04d.h5' % (step+1)
	g_model.save(filename2)
	# save the classifier model
	filename3 = 'c_model_%04d.h5' % (step+1)
	c_model.save(filename3)
	print('>Saved: %s, %s, and %s' % (filename1, filename2, filename3))

# train the generator and discriminator
def train(g_model, d_model, c_model, gan_model, dataset, latent_dim, n_epochs=20, n_batch=100):
	# select supervised dataset
	X_sup, y_sup = select_supervised_samples(dataset)
	print(X_sup.shape, y_sup.shape)
	# calculate the number of batches per training epoch
	bat_per_epo = int(dataset[0].shape[0] / n_batch)
	# calculate the number of training iterations
	n_steps = bat_per_epo * n_epochs
	# calculate the size of half a batch of samples
	half_batch = int(n_batch / 2)
	print('n_epochs=%d, n_batch=%d, 1/2=%d, b/e=%d, steps=%d' % (n_epochs, n_batch, half_batch, bat_per_epo, n_steps))
	# manually enumerate epochs
	for i in range(n_steps):
		# update supervised discriminator (c)
		[Xsup_real, ysup_real], _ = generate_real_samples([X_sup, y_sup], half_batch)
		c_loss, c_acc = c_model.train_on_batch(Xsup_real, ysup_real)
		# update unsupervised discriminator (d)
		[X_real, _], y_real = generate_real_samples(dataset, half_batch)
		d_loss1 = d_model.train_on_batch(X_real, y_real)
		X_fake, y_fake = generate_fake_samples(g_model, latent_dim, half_batch)
		d_loss2 = d_model.train_on_batch(X_fake, y_fake)
		# update generator (g)
		X_gan, y_gan = generate_latent_points(latent_dim, n_batch), ones((n_batch, 1))
		g_loss = gan_model.train_on_batch(X_gan, y_gan)
		# summarize loss on this batch
		print('>%d, c[%.3f,%.0f], d[%.3f,%.3f], g[%.3f]' % (i+1, c_loss, c_acc*100, d_loss1, d_loss2, g_loss))
		# evaluate the model performance every so often
		if (i+1) % (bat_per_epo * 1) == 0:
			summarize_performance(i, g_model, c_model, latent_dim, dataset)

# size of the latent space
latent_dim = 100
# create the discriminator models
d_model, c_model = define_discriminator()
# create the generator
g_model = define_generator(latent_dim)
# create the gan
gan_model = define_gan(g_model, d_model)
# load image data
dataset = load_real_samples()
# train model
train(g_model, d_model, c_model, gan_model, dataset, latent_dim)

---

➡️ **Next / 下一步**: File 7 of 7